In [15]:
# -------------------------------------------------------------------
# Define storage paths (ADLS Gen2)
# -------------------------------------------------------------------

RAW_PATH    =  "abfss://bronze@flightdatalakegen2.dfs.core.windows.net/*.csv"
SILVER_PATH = "abfss://silver@flightdatalakegen2.dfs.core.windows.net/"



StatementMeta(sparkpool1, 18, 6, Finished, Available, Finished, False)

In [16]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

StatementMeta(sparkpool1, 18, 7, Finished, Available, Finished, False)

In [17]:
# -------------------------------------------------------------------
# Load raw dataset (all years 2009–2018)
# -------------------------------------------------------------------

df_raw = spark.read \
    .option("header", "true") \
    .option("nullValue", "") \
    .option("mode", "PERMISSIVE") \
    .csv(RAW_PATH)

print(f"Raw rows loaded: {df_raw.count():,}")
print(f"Columns: {len(df_raw.columns)}")

StatementMeta(sparkpool1, 18, 8, Finished, Available, Finished, False)

Raw rows loaded: 61,556,964
Columns: 28


In [18]:
# -------------------------------------------------------------------
# Remove useless / noisy columns
# -------------------------------------------------------------------

df = df_raw.drop("Unnamed: 27")

print("Dropped junk columns successfully")

StatementMeta(sparkpool1, 18, 9, Finished, Available, Finished, False)

Dropped junk columns successfully


In [19]:
# -------------------------------------------------------------------
# Remove duplicate flights using business keys
# -------------------------------------------------------------------

before = df.count()

df = df.dropDuplicates([
    "FL_DATE",
    "OP_CARRIER",
    "OP_CARRIER_FL_NUM",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME"
])

after = df.count()

print(f"Duplicates removed: {before - after:,}")

StatementMeta(sparkpool1, 18, 10, Finished, Available, Finished, False)

Duplicates removed: 1,039,451


In [20]:
# -------------------------------------------------------------------
# Split dataset into:
# 1. Operated flights → full analysis
# 2. Cancelled flights → separate insights
# -------------------------------------------------------------------
from pyspark.sql import functions as F
df_cancelled = df.filter(F.col("CANCELLED") == 1)
df_operated  = df.filter(F.col("CANCELLED") == 0)

print(f"Operated flights: {df_operated.count():,}")
print(f"Cancelled flights: {df_cancelled.count():,}")

StatementMeta(sparkpool1, 18, 11, Finished, Available, Finished, False)

Operated flights: 59,556,533
Cancelled flights: 960,980


In [21]:
# -------------------------------------------------------------------
# Replace dirty strings with NULL
# -------------------------------------------------------------------

bad_values = ["undefined", "NA", "null", "NULL", "", " "]

df_operated = df_operated.replace(bad_values, None)

StatementMeta(sparkpool1, 18, 12, Finished, Available, Finished, False)

In [22]:
from pyspark.sql import functions as F

# -------------------------------
# 1. CLEAN "undefined" (all columns)
# -------------------------------

df = df.select([
    F.when(
        F.lower(F.trim(F.col(c))).like("%undefined%"),
        None
    ).otherwise(F.col(c)).alias(c)
    for c in df.columns
])

# -------------------------------
# 2. DEFINE NUMERIC COLUMNS
# -------------------------------

numeric_cols = [
    "DEP_DELAY",
    "ARR_DELAY",
    "CARRIER_DELAY",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY"
]

# -------------------------------
# 3. CAST NUMERIC COLUMNS
# -------------------------------

for c in numeric_cols:
    df = df.withColumn(c, F.col(c).cast("double"))

# -------------------------------
# 4. HANDLE NULLS (OPTIONAL)
# -------------------------------

df = df.fillna({
    "DEP_DELAY": 0,
    "ARR_DELAY": 0,
    "CARRIER_DELAY": 0,
    "WEATHER_DELAY": 0,
    "NAS_DELAY": 0,
    "SECURITY_DELAY": 0,
    "LATE_AIRCRAFT_DELAY": 0
})

# -------------------------------
# 5. CHECK RESULT
# -------------------------------

df.printSchema()
df.show(5)

StatementMeta(sparkpool1, 18, 13, Finished, Cancelled, Cancelled, False)

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: string (nullable = true)
 |-- DEP_TIME: string (nullable = true)
 |-- DEP_DELAY: double (nullable = false)
 |-- TAXI_OUT: string (nullable = true)
 |-- WHEELS_OFF: string (nullable = true)
 |-- WHEELS_ON: string (nullable = true)
 |-- TAXI_IN: string (nullable = true)
 |-- CRS_ARR_TIME: string (nullable = true)
 |-- ARR_TIME: string (nullable = true)
 |-- ARR_DELAY: double (nullable = false)
 |-- CANCELLED: string (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: string (nullable = true)
 |-- CRS_ELAPSED_TIME: string (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: string (nullable = true)
 |-- AIR_TIME: string (nullable = true)
 |-- DISTANCE: string (nullable = true)
 |-- CARRIER_DELAY: double (nullable = false)
 |-- WEATHER_DELAY: do

In [ ]:
display(df)

StatementMeta(, 18, -1, Cancelled, , Cancelled, True)

In [ ]:
# -------------------------------------------------------------------
# Drop rows missing essential identifiers
# -------------------------------------------------------------------

df_operated = df_operated.dropna(subset=[
    "FL_DATE",
    "OP_CARRIER",
    "ORIGIN",
    "DEST"
])

StatementMeta(, 18, -1, Cancelled, , Cancelled, True)

In [ ]:
# -------------------------------------------------------------------
# Standardize formats for consistency
# -------------------------------------------------------------------
df_operated = df_operated \
    .withColumn("FL_DATE", F.to_date("FL_DATE", "yyyy-MM-dd")) \
    .withColumn("CANCELLED", F.col("CANCELLED").cast("int")) \
    .withColumn("DIVERTED", F.col("DIVERTED").cast("int")) \
    .withColumn("OP_CARRIER", F.upper(F.trim("OP_CARRIER"))) \
    .withColumn("ORIGIN", F.upper(F.trim("ORIGIN"))) \
    .withColumn("DEST", F.upper(F.trim("DEST")))

StatementMeta(, 18, -1, Cancelled, , Cancelled, True)

In [ ]:
# -------------------------------------------------------------------
# Cap extreme delays (data quality control)
# -------------------------------------------------------------------

df_operated = df_operated \
    .withColumn("ARR_DELAY", F.when(F.col("ARR_DELAY") > 1440, 1440)
                .otherwise(F.col("ARR_DELAY"))) \
    .withColumn("DEP_DELAY", F.when(F.col("DEP_DELAY") > 1440, 1440)
                .otherwise(F.col("DEP_DELAY")))

# Negative AIR_TIME is invalid → set to NULL
df_operated = df_operated.withColumn(
    "AIR_TIME",
    F.when(F.col("AIR_TIME") < 0, None).otherwise(F.col("AIR_TIME"))
)

StatementMeta(, 18, -1, Cancelled, , Cancelled, True)

In [ ]:
# -------------------------------------------------------------------
# Create analysis-ready features
# -------------------------------------------------------------------

# Route column
df_operated = df_operated.withColumn(
    "ROUTE",
    F.concat_ws("-", "ORIGIN", "DEST")
)

# Year column for partitioning
df_operated = df_operated.withColumn(
    "YEAR",
    F.year("FL_DATE")
)

StatementMeta(, 18, -1, Cancelled, , Cancelled, True)

In [ ]:
df_operated = df_operated.withColumn("YEAR", F.year("FL_DATE"))

df_operated.write \
    .mode("overwrite") \
    .option("header", "true") \
    .partitionBy("YEAR") \
    .csv(SILVER_PATH + "operated/")

StatementMeta(, 18, -1, Cancelled, , Cancelled, True)